In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import Markdown, display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder

pd.set_option("display.max_rows",100)
pd.set_option("display.max_columns", 60)
project_dir = Path(r"D:\credit")
result_dir = project_dir / "results" / "XGBOOST"
result_dir.mkdir(parents=True, exist_ok=True)
random_state = 42
threshold = 0.5

In [ ]:
df_clean = pd.read_csv("D:/credit/data/data_processing.csv", index_col="id")

## 1. 衍生特征构造

使用原 notebook 的 data_processing.csv，以 id 为索引。
保留偿债压力、账户结构、信用历史、FICO 均值特征；
删除原始日期、FICO 上下界和常量列 policyCode。

| 特征 | 含义 |
| --- | --- |
| income_nonpositive | 年收入非正标记 |
| annual_loan_income_ratio | 贷款金额 / 期限（年）/ 年收入 |
| annual_revolBal_income_ratio | 循环余额 / 年收入 |
| openAcc_ratio、closeAcc | 开放账户占比、关闭账户数 |
| dis_Year | 放款年份减最早信用年份 |
| ficoRange_mean | FICO 上下界均值 |

In [ ]:

income = df_clean["annualIncome"].where(df_clean["annualIncome"] > 0)
term = df_clean["term"].where(df_clean["term"] > 0)
df_clean["income_nonpositive"] = (df_clean["annualIncome"] <= 0).astype("int8")
df_clean["annual_loan_income_ratio"] = (df_clean["loanAmnt"] / term) / income
df_clean["annual_revolBal_income_ratio"] = (df_clean["revolBal"] / income)


total_accounts = df_clean["totalAcc"].where(df_clean["totalAcc"] > 0)
df_clean["openAcc_ratio"] = (df_clean["openAcc"] / total_accounts)
closed_accounts = (df_clean["totalAcc"] - df_clean["openAcc"])
df_clean["closeAcc"] = closed_accounts.where(closed_accounts >= 0)

credit_years = (df_clean["issueYear"] - df_clean["earliesCreditYear"])
df_clean["dis_Year"] = credit_years.where(credit_years >= 0)

df_clean["ficoRange_mean"] = (df_clean["ficoRangeHigh"] + df_clean["ficoRangeLow"]) / 2

df_clean = df_clean.drop(["ficoRangeHigh","ficoRangeLow","issueDate","earliesCreditLine"],axis=1)
df_clean = df_clean.drop(columns="policyCode").replace([np.inf, -np.inf], np.nan)
display(Markdown(
    f"**数据结果：** 共 {len(df_clean):,} 条记录，违约率 {df_clean['isDefault'].mean():.2%}，"
    f"特征构造后共 {df_clean.shape[1] - 1} 个特征。"
))

## 2. 分层划分样本

训练集70% / 验证集20% / 测试集10%比例，按时间划分，模拟真实世界上线情况。

In [ ]:
# df_trainval, df_test = train_test_split(df_clean, test_size=0.2, random_state=42, stratify=df_clean["isDefault"])
# df_train,df_val = train_test_split(df_trainval, test_size=0.1875, random_state=42, stratify=df_trainval["isDefault"])

# X_train = df_train.drop("isDefault", axis=1)
# y_train = df_train["isDefault"]

# X_val = df_val.drop("isDefault", axis=1)
# y_val = df_val["isDefault"]

# X_test = df_test.drop("isDefault", axis=1)
# y_test = df_test["isDefault"]

In [ ]:
df_clean = df_clean.sort_values("issueYear").reset_index(drop=True)
del df_clean["issueYear"]

n_total=df_clean.shape[0]
split_val=int(n_total*0.7)
split_test=int(n_total*0.9)

df_train=df_clean.iloc[0:split_val]
df_val=df_clean[split_val:split_test]
df_test=df_clean[split_test:]


X_train = df_train.drop("isDefault", axis=1)
y_train = df_train["isDefault"]

X_val = df_val.drop("isDefault", axis=1)
y_val = df_val["isDefault"]

X_test = df_test.drop("isDefault", axis=1)
y_test = df_test["isDefault"]

print(f"训练集有{X_train.shape[0]}条数据")
print(f"验证集有{X_val.shape[0]}条数据")
print(f"测试集有{X_test.shape[0]}条数据")
# X_train_raw = df_train.drop(columns="isDefault")
# y_train = df_train["isDefault"]

# X_val_raw = df_val.drop(columns="isDefault")
# y_val = df_val["isDefault"]

# X_test_raw = df_test.drop(columns="isDefault")
# y_test = df_test["isDefault"]

# print(f"训练集有{X_train_raw.shape[0]}条数据")
# print(f"验证集有{X_val_raw.shape[0]}条数据")
# print(f"测试集有{X_test_raw.shape[0]}条数据")

## 3. 标记缺失值
- 检查发现匿名特征、employmentTitle缺失多
- \>5% 的使用缺失标记
- \<5% 的使用众数填充

In [ ]:
class MissingFlagImputer(BaseEstimator, TransformerMixin):
    def __init__(self, missing_rate_threshold: float = 0.05):
        self.missing_rate_threshold = missing_rate_threshold
        self.fill_mode_ = dict() 
        self.flag_cols_ = set()       

    def fit(self, X: pd.DataFrame, y=None):
        X = X.copy()
        for col in X.columns:
            miss_rate = X[col].isna().mean()
            mode_val = X[col].mode(dropna=True).iloc[0]
            self.fill_mode_[col] = mode_val

            if miss_rate > self.missing_rate_threshold:
                self.flag_cols_.add(col)
        return self
    
    def transform(self, X: pd.DataFrame):
        X = X.copy()

        for col in X.columns:
            if col in self.flag_cols_:
                X[f"{col}_missing_flag"] = X[col].isna().astype("int8")

            X[col] = X[col].fillna(self.fill_mode_[col])

        return X


## 4. 聚合交叉特征：交叉类别与连续变量

一维 CrossFeatureTransformer：按信用等级、就业、住房、用途、地区等类别，
计算贷款金额、利率、收入等连续变量的最大值、最小值和中位数。

二维CrossEntropy2DTransform：交叉类别变量，计算香农熵、最大值、最小值、中位数。

In [ ]:

continuous_variable=["loanAmnt","interestRate","installment","annualIncome","dti","ficoRange_mean","revolBal", "revolUtil"]
categorical_variable=["grade","subGrade","employmentTitle","homeOwnership","verificationStatus","purpose","postCode","regionCode","initialListStatus","applicationType","title"]

class CrossFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, continuous_variable, categorical_variable):
        self.continuous_variable = continuous_variable
        self.categorical_variable = categorical_variable

    def fit(self, X, y=None):
        self.group_maps_ = {}

        for category in self.categorical_variable:
            statistics = (
                X.groupby(category, observed=True)[self.continuous_variable]
                .agg(["max", "min", "median","mean"])
            )

            statistics.columns = [
                f"{category}_{feature}_{statistic}"
                for feature, statistic in statistics.columns
            ]

            self.group_maps_[category] = statistics

        return self

    def transform(self, X):
        result = X.copy()

        for category, statistics in self.group_maps_.items():
            result = result.join(
                statistics,
                on=category,
                how="left",
                validate="many_to_one",
            )

        return result


In [ ]:
cat_2d_pairs = [
    ("grade", "homeOwnership"),
    ("grade", "verificationStatus"),
    ("grade","applicationType"),
    ("grade","purpose"),
    ("subGrade", "homeOwnership"),
    ("subGrade", "verificationStatus"),
    ("subGrade","applicationType"),
    ("subGrade","purpose")
]

def shannon_entropy(p_arr):
    p_arr = p_arr[p_arr > 0]
    return -np.sum(p_arr * np.log2(p_arr))

class CrossEntropy2DTransformer(BaseEstimator, TransformerMixin):
    def __init__(self,
                 cat_pairs,
                 cont_vars,
                 agg_funcs = ["max", "min", "median","mean"],
                 # 是否计算【分组内y的香农熵】
                 use_y_entropy=True
                ):
        self.cat_pairs = cat_pairs
        self.cont_vars = cont_vars
        self.agg_funcs = agg_funcs
        self.use_y_entropy = use_y_entropy
        self.group_maps_ = {}

    def fit(self, X, y=None):
        df_fit = X.copy()
        if self.use_y_entropy:
            if y is None:
                raise ValueError("use_y_entropy=True时fit必须传入y！")
            df_fit["__target__"] = y.values

        for (cat1, cat2) in self.cat_pairs:
            group_key = [cat1, cat2]
            # 1. 二维分组连续变量聚合
            agg_stats = df_fit.groupby(group_key, observed=True)[self.cont_vars].agg(self.agg_funcs)
            agg_stats.columns = [f"{cat1}_{cat2}_{feat}_{stat}" for feat, stat in agg_stats.columns]

            # 2. 计算香农熵（基于标签分布）
            if self.use_y_entropy:
                entropy_df = df_fit.groupby(group_key, observed=True)["__target__"].agg(
                    p0=lambda s: (s==0).mean(),
                    p1=lambda s: (s==1).mean()
                )
                entropy_df["entropy"] = entropy_df[["p0","p1"]].apply(shannon_entropy, axis=1)
                entropy_df = entropy_df[["entropy"]].rename(columns={"entropy":f"{cat1}_{cat2}_y_entropy"})
                # 合并聚合特征 + 熵特征
                stat_df = pd.concat([agg_stats, entropy_df], axis=1)
            else:
                stat_df = agg_stats

            self.group_maps_[(cat1, cat2)] = stat_df
        return self

    def transform(self, X):
        res = X.copy()
        for (cat1, cat2), stat_df in self.group_maps_.items():
            res = res.join(stat_df, on=[cat1, cat2], how="left", validate="many_to_one")
        return res


## 5. 筛选特征值
IV 用于衡量特征对正负样本（好坏样本）的区分能力：IV 值越高，特征预测区分能力通常越强；工程上常用 0.02 作为基础筛选阈值。

- 分箱逻辑：自动区分连续特征与离散特征，连续特征采用等频分箱并做重复边界剔除；离散特征不额外分箱，直接以自身取值作为分箱单元。
- 数值平滑处理：分箱内好坏样本数为 0 时填充 0.5，规避 WOE 计算时对数取 0、除零报错问题。

In [ ]:
class IVFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, iv_threshold=0.02, n_bins=10):
        """
        :param iv_threshold: IV筛选阈值，默认0.02，保留IV >= 002的特征
        :param n_bins: 连续特征分箱数量
        """
        self.iv_threshold = iv_threshold
        self.n_bins = n_bins
        self.selected_cols_ = None  # fit后保存筛选出来的特征列表
        self.iv_result_ = None      # fit后保存全部特征IV结果表

    @staticmethod
    def _calc_iv(col, y, n_bins):
        """单特征IV内部计算函数"""
        df = pd.DataFrame({"x": col, "y": y})
        df = df.dropna()
        if df["x"].nunique() <= 1:
            return 0.0

        # 判断连续/离散：唯一值多就分箱
        if df["x"].nunique() > n_bins:
            df["bin"] = pd.qcut(df["x"], q=n_bins, duplicates="drop")
        else:
            df["bin"] = df["x"]

        # 统计好坏样本
        agg = df.groupby("bin")["y"].agg(["count", "sum"])
        agg.columns = ["total", "bad"]
        agg["good"] = agg["total"] - agg["bad"]

        # 加平滑，避免0分母
        agg["good"] = agg["good"].replace(0, 0.5)
        agg["bad"] = agg["bad"].replace(0, 0.5)

        agg["good_pct"] = agg["good"] / agg["good"].sum()
        agg["bad_pct"] = agg["bad"] / agg["bad"].sum()
        agg["woe"] = np.log(agg["good_pct"] / agg["bad_pct"])
        agg["iv_bin"] = (agg["good_pct"] - agg["bad_pct"]) * agg["woe"]
        iv = agg["iv_bin"].sum()
        return iv

    def fit(self, X: pd.DataFrame, y: pd.Series):
        """fit只在训练集计算IV"""
        iv_dict = {}
        for c in X.columns:
            iv = self._calc_iv(X[c], y, self.n_bins)
            iv_dict[c] = iv
        iv_df = pd.DataFrame(iv_dict.items(), columns=["feature", "IV"])
        iv_df = iv_df.sort_values("IV", ascending=False).reset_index(drop=True)

        self.iv_result_ = iv_df
        self.selected_cols_ = iv_df[iv_df["IV"] >= self.iv_threshold]["feature"].tolist()
        return self

    def transform(self, X: pd.DataFrame):
        """筛选特征，只保留fit阶段选出的列"""
        X = X.copy()
        return X[self.selected_cols_]

## 6.特征编码
- 有序编码；
- 低基数类别：独热编码
- 高基数类别：目标编码。

说明：
- 目标编码的 cv=5 仅用于编码器内部交叉拟合，不进行五次模型训练。
- 预处理器在训练集拟合，验证集和测试集只做转换；数值转为 float32，缺失值由 XGBoost 处理。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, TargetEncoder

ordinal_variable = ["grade", "subGrade"]
low_cardinality = ["homeOwnership", "verificationStatus", "purpose","initialListStatus", "applicationType"]
high_cardinality = ["employmentTitle", "postCode", "regionCode", "title"]

grade_order = list("ABCDEFG")
subgrade_order = [
    f"{grade}{level}"
    for grade in grade_order
    for level in range(1, 6)
]


def build_preprocessor():
    encoder = ColumnTransformer(
        [
            (
                "ordinal",
                OrdinalEncoder(
                    categories=[grade_order, subgrade_order],
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                ),
                ordinal_variable,
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                low_cardinality,
            ),
            (
                "target",
                TargetEncoder(
                    target_type="binary",
                    cv=5,
                    shuffle=True,
                    random_state=random_state,
                    smooth="auto",
                ),
                high_cardinality,
            ),
        ],
        remainder="passthrough",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    pipe = make_pipeline(
        MissingFlagImputer(
            missing_rate_threshold=0.05
        ),

        CrossFeatureTransformer(
            continuous_variable=continuous_variable,
            categorical_variable=categorical_variable,
        ),

        CrossEntropy2DTransformer(
            cat_pairs=cat_2d_pairs,
            cont_vars=continuous_variable,
            use_y_entropy=True
        ),

        encoder,

        IVFeatureSelector(
            iv_threshold=0.02,
            n_bins=10,
        ),
    )
    return pipe

preprocessor = build_preprocessor()
X_train = preprocessor.fit_transform(X_train, y_train).astype("float32")
X_val = preprocessor.transform(X_val).astype("float32")
X_test = preprocessor.transform(X_test).astype("float32")
display(Markdown(f"**特征结果：** 编码后共有 **{X_train.shape[1]}** 个特征。"))

## 7. 筛选特征：相关性与VIF

- 先删除全训练集常量或全缺失列；
- 抽样20000条，筛选优先项：
    - IV大
    - 缺失率低
    - 与其他特征相关系数不超过0.8
- 再迭代删除VIF大于10的特征，每删一列重新计算；VIF相同时优先删除IV较低者。临时使用训练抽样中位数填补并标准化，按含截距的线性辅助回归口径计算；此填补不覆盖模型输入。抽样中无变化的稀有列VIF不可估计，不据此删除全训练集中有变化的列。
    （完整独热编码、确定性衍生关系可能产生无限VIF，先消除这类线性依赖）

In [ ]:
import pandas as pd
import numpy as np

def feature_collinearity_filter(
    X_train,
    iv_selector,
    correlation_threshold ,
    vif_threshold,
    selection_sample_size,
    random_state
):
    """
    基于IV排序 + Spearman相关 + VIF迭代剔除多重共线性
    :param X_train: 原始训练集特征
    :param iv_selector: 已经fit好的 IVFeatureSelector 实例（从pipeline取出）
    :return: selected_features, collinearity_removed, final_vif
    """
    # 1. 拿到IV分数，构造排序表
    iv_scores = iv_selector.iv_result_.set_index("feature")["IV"]
    selection_rank = pd.DataFrame({
        "IV": iv_scores.reindex(X_train.columns),
        "missing_rate": X_train.isna().mean(),
        "unique_count": X_train.nunique(dropna=True),
    }).sort_values(["IV", "missing_rate"], ascending=[False, True], kind="stable")

    # 2. 抽样计算相关系数
    sample = X_train.sample(n=min(selection_sample_size, len(X_train)), random_state=random_state)
    # 抽样严格不能非空观测对>100才计算相关系数。用于过滤样本太少、可信度低的相关系数
    corr = sample.corr(method="spearman", min_periods=min(100, len(sample)))

    kept_features = []
    collinearity_removed = []

    # 3. 高相关剔除（优先保留IV更高的）
    for feat in selection_rank.index:
        if selection_rank.loc[feat, "unique_count"] <= 1:
            collinearity_removed.append((feat, "常量或全缺失", "", np.nan))
            continue
        corr_with_kept = corr.loc[feat, kept_features].abs()
        # 判断当前feat与留下来的feats是否大于阈值
        if corr_with_kept.ge(correlation_threshold).any():
            rep = corr_with_kept.idxmax()#删除相关系数最高的kept_feat
            collinearity_removed.append((feat, "高相关", rep, corr_with_kept[rep]))
        else:
            kept_features.append(feat)

    # 4. VIF迭代剔除
    def calc_vif(df):
        vals = df.astype("float64").fillna(df.median())
        std = vals.std(ddof=0)

        #筛选有效变量
        varying = (std > 1e-12) & std.notna()
        if varying.sum() <= 1:
            return pd.Series(np.nan, index=df.columns, name="VIF")

        # 标准化
        z = (vals.loc[:, varying] - vals.loc[:, varying].mean()) / std[varying]
        moment = z.T @ z / len(z) #相关矩阵

        #求特征值
        eigval, eigvec = np.linalg.eigh(moment)
        pos = eigval > 1e-10


        vif = (eigvec[:, pos] ** 2 / eigval[pos]).sum(axis=1)

        vif_ser = pd.Series(np.nan, index=df.columns, name="VIF")
        vif_ser.loc[varying] = vif
        mask_inf = ((eigvec[:, ~pos] ** 2).sum(axis=1)) > 1e-8
        idx_varying = vif_ser.index[varying]
        vif_ser.loc[idx_varying[mask_inf]] = np.inf

        return vif_ser
        # vif_ser.loc[varying][((eigvec[:, ~pos] ** 2).sum(axis=1)) > 1e-8] = np.inf
        # return vif_ser

    while len(kept_features) > 1:
        vif_ser = calc_vif(sample[kept_features])
        high_vif = vif_ser[vif_ser > vif_threshold]
        if high_vif.empty:
            break
        # VIF大优先删，同VIF删IV低的
        drop_rank = pd.DataFrame({"VIF": high_vif, "IV": iv_scores.reindex(high_vif.index)})
        drop_rank = drop_rank.sort_values(["VIF", "IV"], ascending=[False, True], kind="stable")
        drop_feat = drop_rank.index[0]
        collinearity_removed.append((drop_feat, "VIF超限迭代删除", "", drop_rank.loc[drop_feat, "VIF"]))
        kept_features.remove(drop_feat)

    final_vif = calc_vif(sample[kept_features])
    return kept_features, collinearity_removed, final_vif, sample, corr


In [ ]:
##vif计算替换
# from statsmodels.stats.outliers_influence import variance_inflation_factor

# def calc_vif_sm(df):
#     vals = df.astype("float64").fillna(df.median())
#     vif_list = []
#     for i in range(vals.shape[1]):
#         v = variance_inflation_factor(vals.values, i)
#         vif_list.append(v)
#     return pd.Series(vif_list, index=df.columns, name="VIF")

# # -------- 替换你原来的while循环 --------
# while len(kept_features) > 1:
#     current_vif = calc_vif_sm(selection_sample[kept_features])
#     high_vif = current_vif[current_vif > vif_threshold]
#     if high_vif.empty:
#         break
#     removal_rank = pd.DataFrame({"VIF": high_vif, "IV": iv_scores.reindex(high_vif.index)})
#     removal_rank = removal_rank.sort_values(["VIF", "IV"], ascending=[False, True], kind="stable")
#     feature = removal_rank.index[0]
#     collinearity_removed.append((feature, "VIF超阈值，迭代删除后重算", "", removal_rank.loc[feature, "VIF"]))
#     kept_features.remove(feature)


In [ ]:
# 从pipeline取出已经fit完成的iv selector
iv_selector = next(stage for _, stage in preprocessor.steps if isinstance(stage, IVFeatureSelector))

selected_features, collinearity_log, final_vif, selection_sample, selection_corr= feature_collinearity_filter(
    X_train,
    iv_selector=iv_selector,
    correlation_threshold=0.80,
    vif_threshold=10.0,
    selection_sample_size=20000,
    random_state=42
)

# 切分数据集
X_train = X_train.loc[:, selected_features]
X_val   = X_val.loc[:, selected_features]
X_test  = X_test.loc[:, selected_features]

print(f"训练前特征筛选：{len(iv_selector.selected_cols_)} → {X_train.shape[1]} 个特征")


### 相关系数图

使用同一训练抽样绘制全部保留特征的Spearman相关系数，展示筛选后仍存在的关联结构

In [ ]:
selection_sample = X_train.sample(n=min(selection_sample.size, len(X_train)), random_state=random_state)
selection_corr = selection_sample.corr(method="spearman", min_periods=min(100, len(selection_sample)))

feature_ids = {f: f"F{i+1:03d}" for i, f in enumerate(selected_features)}
final_corr = selection_corr.loc[selected_features, selected_features]

fig, ax = plt.subplots(figsize=(12,10))
im = ax.imshow(final_corr, cmap="RdBu_r", vmin=-1, vmax=1)

step = max(1, int(np.ceil(len(selected_features)/35)))
ticks = np.arange(0, len(selected_features), step)
labels = [feature_ids[selected_features[p]] for p in ticks]
ax.set_xticks(ticks, labels, rotation=90, fontsize=8)
ax.set_yticks(ticks, labels, fontsize=8)


### 保留特征说明

按实际构造将保留列分为原始数值、匿名特征、业务衍生、有序编码、独热编码、目标编码、一维聚合、二维聚合、标签熵及缺失标记；不存在的类别不显示。表中列出全部保留特征，而不是只展示Top变量。

保留理由是：先达到当前IV阈值，再未被本次高相关和VIF规则删除；不等于已证明独立增量价值。业务解释说明变量为何可能与信用风险相关，不预设效应方向；匿名字段不编造含义，类别码不猜测映射，标签熵不等于违约率。各组IV区间为实际保留列的最小至最大值，最终VIF为同一训练抽样的估计值。

In [ ]:
import pandas as pd

# 基础业务字段字典：原始特征 -> (大类, 名称, 简要说明)
business_fields = {
    'loanAmnt': ('贷款与定价', '贷款金额', '融资规模与债务负担'),
    'term': ('贷款与定价', '贷款期限', '借款周期'),
    'interestRate': ('贷款与定价', '贷款利率', '贷款定价信息'),
    'installment': ('偿债能力', '月供', '每期还款压力'),
    'grade': ('信用评级', '贷款等级', '风险分层标记'),
    'subGrade': ('信用评级', '子等级', '细分风险分层'),
    'employmentTitle': ('借款人属性', '就业岗位', '职业群体特征'),
    'employmentLength': ('借款人属性', '就业年限', '就业稳定性'),
    'homeOwnership': ('借款人属性', '住房产权', '资产情况'),
    'annualIncome': ('偿债能力', '年收入', '收入与偿债来源'),
    'verificationStatus': ('申请与核验', '收入核验状态', '信息可信度'),
    'purpose': ('申请与核验', '贷款用途', '资金使用场景'),
    'postCode': ('地域信息', '邮编', '地域差异'),
    'regionCode': ('地域信息', '地区编码', '区域环境差异'),
    'dti': ('偿债能力', '债务收入比', '相对负债水平'),
    'delinquency_2years': ('信用历史与账户', '近两年逾期次数', '历史逾期行为'),
    'openAcc': ('信用历史与账户', '在用信贷账户数', '信贷活跃规模'),
    'pubRec': ('信用历史与账户', '公共负面记录', '不良记录标记'),
    'pubRecBankruptcies': ('信用历史与账户', '破产记录', '破产相关记录'),
    'revolBal': ('偿债能力', '循环贷余额', '循环债务存量'),
    'revolUtil': ('偿债能力', '循环额度使用率', '额度占用紧张度'),
    'totalAcc': ('信用历史与账户', '总信贷账户数', '信贷历史广度'),
    'initialListStatus': ('申请与核验', '列表状态', '放款批次标记'),
    'applicationType': ('申请与核验', '申请类型', '申请主体类型'),
    'title': ('申请与核验', '贷款标题', '贷款描述标记'),
    'issueYear': ('时间批次', '放款年份', '放款时间批次'),
    'earliesCreditYear': ('信用历史与账户', '最早授信年份', '信用起始时间'),
    'income_nonpositive': ('偿债能力', '非正收入标记', '收入异常标记'),
    'annual_loan_income_ratio': ('偿债能力', '本金收入比', '本金偿还压力'),
    'annual_revolBal_income_ratio': ('偿债能力', '循环余额收入比', '循环债务相对规模'),
    'openAcc_ratio': ('信用历史与账户', '在用账户占比', '账户结构占比'),
    'closeAcc': ('信用历史与账户', '已关闭账户数', '历史销户数量'),
    'dis_Year': ('信用历史与账户', '信用年限', '信用历史时长'),
    'ficoRange_mean': ('信用评级', 'FICO得分', '客户信用分代理变量'),
}
statistic_names = {"max": "最大值", "min": "最小值", "median": "中位数", "mean": "均值"}

# 聚合特征大类说明，统一话术
feature_explanations = {}
one_dimensional = next(stage for _, stage in preprocessor.steps if isinstance(stage, CrossFeatureTransformer))
for cat, stats_df in one_dimensional.group_maps_.items():
    for feat in stats_df.columns:
        feature_explanations[feat] = ("一维分组聚合", "单维度分组统计量，刻画同组客户群体分布特征")

for first, second in cat_2d_pairs:
    for stat in statistic_names.keys():
        feat = f"{first}_{second}_{stat}"
        feature_explanations[feat] = ("二维分组聚合", "双维度交叉分组统计量，刻画交叉群体分布特征")
    feature_explanations[f"{first}_{second}_y_entropy"] = ("二维标签熵", "交叉分组内标签混杂度，衡量分组区分度")

# 各类编码特征统一说明
encoder_stage = next(stage for _, stage in preprocessor.steps if isinstance(stage, ColumnTransformer))
for encoder_name, transformer, columns in encoder_stage.transformers_:
    if isinstance(transformer, OneHotEncoder):
        for feat in transformer.get_feature_names_out(columns):
            feature_explanations[feat] = ("独热编码", "类别变量0-1编码，表征分类归属")
    elif isinstance(transformer, OrdinalEncoder):
        for feat in columns:
            feature_explanations[feat] = ("有序编码", "有序类别数值化分层")
    elif isinstance(transformer, TargetEncoder):
        for feat in columns:
            feature_explanations[feat] = ("目标编码", "基于标签平滑编码，压缩高基数类别")

business_derived = {"income_nonpositive", "annual_loan_income_ratio", "annual_revolBal_income_ratio", "openAcc_ratio", "closeAcc", "dis_Year", "ficoRange_mean"}

# 生成明细
feature_rows = []
for feat in selected_features:
    if feat in feature_explanations:
        group, meaning = feature_explanations[feat]
    elif feat.endswith("_missing_flag"):
        group, meaning = "缺失标记", "原始变量缺失指示标记，刻画缺失模式"
    elif feat.startswith("n") and feat[1:].isdigit():
        group, meaning = "匿名特征", "匿名变量，业务含义未披露"
    else:
        group = "业务衍生" if feat in business_derived else "原始数值"
        _, name, desc = business_fields.get(feat, ("待确认", feat, "含义待字典补充"))
        meaning = f"{name}；{desc}"

    feature_rows.append({
        "编号": feature_ids[feat],
        "feature": feat,
        "特征类别": group,
        "业务解释": meaning,
        "IV": iv_selector.iv_result_.set_index("feature")["IV"][feat],
        "最终VIF": final_vif[feat],
        "保留原因": f"IV≥{iv_selector.iv_threshold:g}，通过相关性与VIF共线性筛选"
    })
detail_df = pd.DataFrame(feature_rows)

# 大类汇总，使用换行符 \n，jupyter表格内自动换行
summary_rows = []
for group, sub_df in detail_df.groupby("特征类别", sort=False):
    lines = [f"{r['编号']} {r['feature']}：{r['业务解释']}" for _, r in sub_df.iterrows()]
    text = "\n".join(lines)
    summary_rows.append({
        "特征类别": group,
        "数量": len(sub_df),
        "IV区间": f"{sub_df['IV'].min():.4f} ~ {sub_df['IV'].max():.4f}",
        "特征清单&说明": text,
        "筛选依据": f"IV阈值{iv_selector.iv_threshold:g}，Spearman相关阈值0.8，VIF阈值10"
    })
summary_df = pd.DataFrame(summary_rows)

# 直接在notebook输出表格
print("===== 特征大类汇总 =====")
display(summary_df)

print("\n===== 特征明细 =====")
display(detail_df)

# 导出csv，按需打开
# detail_df.to_csv(selection_output_dir / "selected_feature_details.csv", index=False, encoding="utf-8-sig")
# summary_df.to_csv(selection_output_dir / "selected_feature_group_summary.csv", index=False, encoding="utf-8-sig")


In [ ]:
from pathlib import Path
import json

feature_dir = Path(r"D:\credit\data\xgboost_features")
feature_dir.mkdir(parents=True, exist_ok=True)

feature_columns = X_train.columns.tolist()
datasets = {
    "train": (X_train, y_train),
    "val": (X_val, y_val),
    "test": (X_test, y_test),
}

for split_name, (features, target) in datasets.items():
    if features.columns.tolist() != feature_columns:
        raise ValueError(f"{split_name} 的特征名称或顺序与训练集不一致")
    if not features.index.equals(target.index):
        raise ValueError(f"{split_name} 的特征与标签索引不一致")
    if "isDefault" in features.columns:
        raise ValueError(f"{split_name} 的特征中不应包含标签 isDefault")

    output_path = feature_dir / f"{split_name}.parquet"
    features.assign(isDefault=target).to_parquet(output_path, index=True)
    print(f"已保存 {split_name}：{len(features):,} 条，{features.shape[1]} 个特征")

metadata = {"target": "isDefault", "feature_columns": feature_columns}
(feature_dir / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)